# Étape 2

Étape 2 : Étape de nettoyage Dans cette étape, on s’intéresse à implémenter les correctifs soulignés dans l’étape 1. De ce fait, il serait important de considérer les opérations suivantes : 

* Imputing: évaluer les colonnes avec des valeurs manquantes. Par exemple, voir les colonnes: 
* title, revol_util et pub_rec_bankruptcies Une stratégie à employer: Supprimer la colonne ayant plus de 1 à 2% de valeurs manquantes Supprimer les lignes ayant des valeurs NaN  

* Convertir les colonnes catégorielles en numériques. Faire attention ici aux valeurs ordinales et nominales (dummy var).  

* Suppression de colonnes non adéquates pour la prédiction  

* Suppression des colonnes qui ne seront intéressantes que pour prédire le statut de paiement du prêt. Les colonnes dont les valeurs seront obtenues après le prêt ne doivent pas être considérées. Par exemple, évaluer si les colonnes suivantes sont à supprimer:  

	* zip_code  
	* out_prncp  
	* out_prncp_inv  
	* total_pymnt  
	* total_pymnt_inv  
	* total_rec_prncp  
	* total_rec_int  
	* total_rec_late_fee  
	* recoveries  
	* collection_recovery_fee  
	* last_pymnt_d  
	* last_pymnt_amnt  

Vérifier si la colonne cible est dans un format adéquat pour le modèle.  

Correction/Standardisation/Normalisation de données  

In [109]:

import pandas as pd 
import numpy as np
from utils.utils import distributional_summary, degree_completeness, degree_validity, get_serie_type, get_df_types, SerieValidityMapper
import matplotlib.pyplot as plt 
from typing import Dict, List, Callable, Any, Tuple


df_data = pd.read_csv('data/lending_club_loans.csv') 
labels = pd.read_excel('data/lending_club_data_dic.xlsx') 
labels.index = labels['LoanStatNew']
labels = labels.drop(columns=['LoanStatNew'])
labels = labels.loc[df_data.columns] # ! limits labels to our variables of interests


def convert_percentage(serie:pd.Series):
  return [ float(str(v).replace('%', '')) for v in serie.values]

def convert_date(serie:pd.Series): 
  return pd.to_datetime(serie, format='%b-%y') 

def delta_month(d1:pd.Series, d2:pd.Series):
	return ( (d1.dt.year - d2.dt.year) * 12 + (d1.dt.month - d2.dt.month) ).astype('Int64')

def reverse_encoding(serie:pd.Series, encoding:dict): 
	reversed = { v:k for k,v in encoding.items() } # ! reverse to encoding to get conversion FROM cat -> num instead of num -> cat 
	return [ reversed.get(v, np.nan) for v in serie.values ]


summary = distributional_summary(df_data) 
varname_id = ['id', 'member_id'] 
varname_cardinality_1 = summary.loc[:, summary.loc['cardinality'] == 1].columns # ! to document. 
varname_date = ['last_pymnt_d', 'last_credit_pull_d', 'earliest_cr_line', 'issue_d'] 
varname_percentage = ['int_rate', 'revol_util'] 


In [110]:

df = df_data.copy() 
# ! subgrade 
#df['_sub_grade'] = [ int(str(v)[1]) for v in df['sub_grade']] 
#df['_sub_grade'].unique() 

cat_var = ['term', 'grade', 'sub_grade', 'home_ownership', 'verification_status', 'loan_status', 'addr_state'] 

encodings = {
  'emp_length':{
    0: '< 1 year',
    1: '1 year',
    2: '2 years',
    3: '3 years',
    4: '4 years',
    5: '5 years',
    6: '6 years',
    7: '7 years',
    8: '8 years',
    9: '9 years',
    10: '10+ years'}
}
for c in cat_var:
	encodings[c] = { i:v for i,v in enumerate(sorted(df[c].unique()))}


In [111]:
df_data #[varname_date] 
#labels.loc[varname_date]

,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,collection_recovery_fee,last_pymnt_d,last_pymnt_amnt,last_credit_pull_d,collections_12_mths_ex_med,application_type,chargeoff_within_12_mths,delinq_amnt,pub_rec_bankruptcies,tax_liens
0,1077501,1296599,5000,5000,4975.0,36 months,10.65%,162.87,B,B2,...,0.00,Jan-15,171.62,Jan-17,0.0,INDIVIDUAL,0.0,0,0.0,0.0
1,1077430,1314167,2500,2500,2500.0,60 months,15.27%,59.83,C,C4,...,1.11,Apr-13,119.66,Oct-16,0.0,INDIVIDUAL,0.0,0,0.0,0.0
2,1077175,1313524,2400,2400,2400.0,36 months,15.96%,84.33,C,C5,...,0.00,Jun-14,649.91,Jan-17,0.0,INDIVIDUAL,0.0,0,0.0,0.0
3,1076863,1277178,10000,10000,10000.0,36 months,13.49%,339.31,C,C1,...,0.00,Jan-15,357.48,Apr-16,0.0,INDIVIDUAL,0.0,0,0.0,0.0
4,1075358,1311748,3000,3000,3000.0,60 months,12.69%,67.79,B,B5,...,0.00,Jan-17,67.30,Jan-17,0.0,INDIVIDUAL,0.0,0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39781,92187,92174,2500,2500,1075.0,36 months,8.07%,78.42,A,A4,...,0.00,Jul-10,80.90,Jun-10,NaN,INDIVIDUAL,NaN,0,NaN,NaN
39782,90665,90607,8500,8500,875.0,36 months,10.28%,275.38,C,C1,...,0.00,Jul-10,281.94,Jul-10,NaN,INDIVIDUAL,NaN,0,NaN,NaN
39783,90395,90390,5000,5000,1325.0,36 months,8.07%,156.84,A,A4,...,0.00,Apr-08,0.00,Jun-07,NaN,INDIVIDUAL,NaN,0,NaN,NaN
39784,90376,89243,5000,5000,650.0,36 months,7.43%,155.38,A,A2,...,0.00,Jan-08,0.00,Jun-07,NaN,INDIVIDUAL,NaN,0,NaN,NaN


In [112]:

# ! Remove some columns
df_transformed = df_data.drop(columns=[*varname_id, *varname_cardinality_1]).copy()


# ? Template for transformation function. Helps intellisense and structures
type TransformerFunc = Callable[[pd.DataFrame, str], pd.Series] 

transform_mapper:Dict[str, Tuple[TransformerFunc, str]] = { 
	# ! convert string percentage -> numerical percentage 
  'int_rate': (lambda df,c: convert_percentage(df[c]), None), 
  'revol_util' : (lambda df,c: convert_percentage(df[c]), None), 
  
  # ! convert string date -> datetime values 
  'last_pymnt_d':	(lambda df,c: convert_date(df[c]), None), 
  'last_credit_pull_d': (lambda df,c: convert_date(df[c]), None), 
  'earliest_cr_line': (lambda df,c: convert_date(df[c]), None), 
  'issue_d': (lambda df,c: convert_date(df[c]), None), 
  
  # ! emp_length to reverse encoding 
  'emp_length': (lambda df,c: reverse_encoding(df[c], encodings[c]), 'emp_length_num'), 
} 

imputations_mapper:Dict[str, Tuple[TransformerFunc, str]] = { 
	# ! Impute mean values for ORDINAL variable "emp_length" instead of using most frequent value. 
	'emp_length_num': (lambda df, c: int(round(df[c].mean(), 0)), 'emp_length_impute'), 
	'emp_length': (lambda df, c: df['emp_length_impute'].map(encodings['emp_length']), 'emp_length'), 
	
	# ! Impute mean and round to integer value
	'revol_util': (lambda df, c: int(round(df[c].mean(), 0)), 'revol_util'), 
	'pub_rec_bankruptcies': (lambda df, c: int(round(df[c].mean(), 0)), 'pub_rec_bankruptcies'), 
} 

flag_mapper:Dict[str, Tuple[TransformerFunc, str]] = { 
	# ! Flag missing date values instead of imputations 
	'last_pymnt_d': (lambda df, c: pd.notnull(df[c]), 'last_pymnt_d_flag'), # flag null values 
	'last_credit_pull_d': (lambda df, c: pd.notnull(df[c]), 'last_credit_pull_d_flag'), # flag null values 
} 

# ! Transformations 
for varname, (func, outvarname) in transform_mapper.items(): 
	outvarname = outvarname or varname 
	df_transformed[outvarname] = func(df_transformed, varname) 

# ! Imputations 
for varname, (func, outvarname) in imputations_mapper.items(): 
	outvarname = outvarname or varname 
	df_transformed[outvarname] = df_transformed[varname].fillna(func(df_transformed, varname)) 


# ? Not necessary if we drop date columns 
# ! Flagging
for varname, (func, outvarname) in flag_mapper.items(): 
	outvarname = outvarname or varname 
	df_transformed[outvarname] = func(df_transformed, varname) 


# ! Drop flagged observations 
varname_flag = ['last_pymnt_d_flag', 'last_credit_pull_d_flag'] #  
to_keep = df_transformed[varname_flag].all(axis=1) 
df_transformed['to_keep'] = to_keep
df_rejected = df_transformed[~to_keep]  # ? Keep rejected observations for documentation purposes 
df_transformed = df_transformed[to_keep] 

# ! Remove temporary variables 
df_transformed = df_transformed.drop(columns=[*varname_flag, 'emp_length_num', 'emp_length_impute', 'to_keep']) 

## Dummies here ?

## Standardisation 

## Distributional summary (Apres transformation)

In [113]:
summary = distributional_summary(df_transformed) 
summary 

,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_length,home_ownership,...,total_pymnt_inv,total_rec_prncp,total_rec_int,total_rec_late_fee,recoveries,collection_recovery_fee,last_pymnt_d,last_pymnt_amnt,last_credit_pull_d,pub_rec_bankruptcies
type,numerical,numerical,numerical,string,numerical,numerical,string,string,string,string,...,numerical,numerical,numerical,numerical,numerical,numerical,date,numerical,date,numerical
dtype,int64,int64,float64,object,float64,float64,object,object,object,object,...,float64,float64,float64,float64,float64,float64,datetime64[ns],float64,datetime64[ns],float64
N,39713,39713,39713,39713,39713,39713,39713,39713,39713,39713,...,39713,39713,39713,39713,39713,39713,39713,39713,39713,39713
count,39713,39713,39713,39713,39713,39713,39713,39713,39713,39713,...,39713,39713,39713,39713,39713,39713,39713,39713,39713,39713
missing_p,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
cardinality,885,1042,8215,2,371,15387,7,35,11,5,...,37500,6919,35044,1364,4518,2644,109,35240,114,3
unique_p,0.00491,0.004432,0.179589,0.0,0.000428,0.228464,0.0,0.0,0.0,0.0,...,0.915166,0.150706,0.810994,0.033087,0.100395,0.054567,0.000025,0.797472,0.000151,0.0
most_freq,10000,10000,5000.0,36 months,10.99,311.11,B,B3,10+ years,RENT,...,0.0,10000.0,1196.57,0.0,0.0,0.0,2013-03-01 00:00:00,200.0,2017-01-01 00:00:00,0.0
least_freq,15325,1125,9994.464812,60 months,16.71,255.43,G,G5,9 years,NONE,...,13909.62,721.02,3268.68,19.89,3216.34,337.8,2008-02-01 00:00:00,1388.77,2008-06-01 00:00:00,2.0
mean,11237.001737,10963.862589,10415.505431,NaN,12.024434,324.870992,NaN,NaN,NaN,NaN,...,11663.690633,9873.357015,2280.492476,1.390898,95.772386,12.516082,NaN,2684.008664,NaN,0.042455


In [114]:
v_completeness = degree_completeness(df_transformed) 
h_completeness = degree_completeness(df_transformed, axis=1) 
v_completeness.sort_values(by='completeness') 



,completeness
loan_amnt,1.0
funded_amnt,1.0
funded_amnt_inv,1.0
term,1.0
int_rate,1.0
installment,1.0
grade,1.0
sub_grade,1.0
emp_length,1.0
home_ownership,1.0


## Graph distribution (Apres transformation)